In [0]:
%run ./professionaludemycontent/Databricks-Certified-Data-Engineer-Professional-main/Includes/Copy-Datasets

In [0]:
data_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
data_schema = spark.sql("SELECT current_schema()").collect()[0][0]

In [0]:
%sql
use catalog 'profudemy'

In [0]:
data_catalog

In [0]:
%sql
use schema bookstore_eng_pro;

In [0]:
data_schema

In [0]:
files = dbutils.fs.ls(f"{bookstore.dataset_path}/kafka-raw")
display(files)

In [0]:
def process_bronze():
    schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

    query = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format","json")
                        .schema(schema)
                        .load(f"{bookstore.dataset_path}/kafka-raw")
                        .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))  
                        .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
            .writeStream
                        .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                        .option("mergeSchema", True)
                        .partitionBy("topic", "year_month")
                        .trigger(availableNow=True)
                        .table("bronze"))

In [0]:
process_bronze()

In [0]:
%sql
select count(*) from bronze;

In [0]:
%sql
select cast(key as string), cast(value as string)
from bronze limit 2;

In [0]:
%sql
select distinct(topic) from bronze;

In [0]:
%sql
select v.*
from (
select from_json(cast(value as string),  "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
from bronze
where topic="orders"
) limit 2;

In [0]:
#streaming temporary view
(spark.readStream
      .table("bronze")
      .createOrReplaceTempView("bronze_tmp"))

In [0]:
%sql
select count(*) from bronze where topic="orders";

In [0]:
# Explicitly set the checkpoint location for streaming display, as implicit temporary checkpoint locations are not supported in the current workspace.

orders_silver_df = spark.sql("""
    SELECT v.*
    FROM (
    SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
    FROM bronze_tmp
    WHERE topic = "orders")
""")

display(orders_silver_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/orders_silver_{time.time()}")

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_silver_tmp AS
  SELECT v.*
  FROM (
    SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
    FROM bronze_tmp
    WHERE topic = "orders")

In [0]:
#classic append mode
query = (spark.table("orders_silver_tmp")
               .writeStream
               .option("checkpointLocation", f"{bookstore.checkpoint_path}/orders_silver")
               .trigger(availableNow=True)
               .table("orders_silver"))

query.awaitTermination()

In [0]:
%sql
select count(*) from orders_silver;

Refactor Everything to use with Python API

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
        .select("v.*")
     .writeStream
        .option("checkpointLocation", f"{bookstore.checkpoint_path}/orders_silver_py")
        .trigger(availableNow=True)
        .table("orders_silver_py"))

query.awaitTermination()

In [0]:
%sql
select count(*) from orders_silver_py;